# 第一次独立训练 SVM
作者：赵知行

目标：从内置Iris数据出发，完整执行划分、训练折预处理、交叉验证、最后测试与Plotly绘图。仅依赖Python，不读取任何项目脚本、CSV或R会话；可以复制到任意目录运行。

## 环境
在VS Code选择Python内核，点击全部运行。依赖见同目录requirements.txt；可在终端执行 `python -m pip install -r requirements.txt`。Plotly结果使用Notebook MIME，VS Code需Jupyter扩展及支持Plotly的Notebook渲染器。

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report


## 1. 载入数据，先保留测试集
Iris共150个样本、4个数值特征、3类。使用全部4维训练；本例不把二维图当成四维边界。行ID用于检查划分，不能输入模型。

In [2]:
iris = load_iris()
X, y = iris.data, iris.target
train_ids, test_ids = train_test_split(np.arange(len(y)), test_size=0.3,
                                     stratify=y, random_state=42)
X_train, X_test = X[train_ids], X[test_ids]
y_train, y_test = y[train_ids], y[test_ids]
assert not set(train_ids) & set(test_ids)
print('训练样本:', len(train_ids), '测试样本:', len(test_ids))
pd.DataFrame(X_train[:5], columns=iris.feature_names)


训练样本: 105 测试样本: 45


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,2.5,3.0,1.1
1,6.2,2.2,4.5,1.5
2,5.1,3.8,1.5,0.3
3,6.8,3.2,5.9,2.3
4,5.7,2.8,4.1,1.3


## 2. 把预处理放进Pipeline
每个训练折只用自己的样本拟合StandardScaler，验证折与测试集只应用变换。提前对全部数据标准化会泄漏。

In [3]:
pipeline = Pipeline([('scale', StandardScaler()), ('svm', SVC(kernel='rbf'))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
parameter_grid = {'svm__C': [0.1, 1, 10], 'svm__gamma': [0.01, 0.1, 1]}
search = GridSearchCV(pipeline, parameter_grid, cv=cv,
                      scoring='f1_macro', n_jobs=1, return_train_score=True)
search.fit(X_train, y_train)
print('最佳参数:', search.best_params_)
print('最佳训练集内CV macro-F1:', search.best_score_)


最佳参数: {'svm__C': 10, 'svm__gamma': 0.01}
最佳训练集内CV macro-F1: 0.980854700854701


## 3. 查看CV结果，理解选参
下表包括每个配置的训练与验证均值、验证标准差。最高均值是选参依据，不能把它当成无偏的泛化估计；网格边缘最优时，可在训练数据内规划扩展搜索。

In [4]:
cv_results = pd.DataFrame(search.cv_results_)
cv_table = cv_results[['param_svm__C', 'param_svm__gamma',
                       'mean_train_score', 'mean_test_score', 'std_test_score']]
cv_table.sort_values('mean_test_score', ascending=False)


,param_svm__C,param_svm__gamma,mean_train_score,mean_test_score,std_test_score
6,10.0,0.01,0.985713,0.980855,0.023448
7,10.0,0.10,0.990469,0.971380,0.038120
4,1.0,0.10,0.988091,0.971282,0.023448
5,1.0,1.00,0.980943,0.961807,0.035661
8,10.0,1.00,1.000000,0.933187,0.023209
2,0.1,1.00,0.955013,0.904280,0.053544
3,1.0,0.01,0.887258,0.874620,0.049152
1,0.1,0.10,0.882736,0.855182,0.060593
0,0.1,0.01,0.880010,0.855078,0.029588


In [5]:
heat = cv_table.pivot(index='param_svm__C', columns='param_svm__gamma', values='mean_test_score')
fig_cv = go.Figure(go.Heatmap(z=heat.to_numpy(), x=heat.columns.astype(str),
                            y=heat.index.astype(str), colorscale='Viridis',
                            hovertemplate='C=%{y}, γ=%{x}<br>CV macro-F1=%{z:.3f}<extra></extra>'))
fig_cv.update_layout(title='训练集内五折验证：C与γ', xaxis_title='γ', yaxis_title='C',
                     height=550, width=1000)
fig_cv.show(renderer='plotly_mimetype')


## 4. 参数确定后，才看测试结果
测试集没有参与拟合、预处理与选参。反复看测试分数再改参数会使它失去最终评估的意义。下面的数值只对应这一次固定划分。

In [6]:
prediction = search.predict(X_test)
test_accuracy = accuracy_score(y_test, prediction)
test_macro_f1 = f1_score(y_test, prediction, average='macro')
print('测试accuracy:', test_accuracy)
print('测试macro-F1:', test_macro_f1)
print('支持向量数:', search.best_estimator_.named_steps['svm'].n_support_)
print(classification_report(y_test, prediction, target_names=iris.target_names, zero_division=0))


测试accuracy: 0.9111111111111111
测试macro-F1: 0.9107142857142857
支持向量数: [ 3 19 16]
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        15
  versicolor       0.82      0.93      0.88        15
   virginica       0.92      0.80      0.86        15

    accuracy                           0.91        45
   macro avg       0.92      0.91      0.91        45
weighted avg       0.92      0.91      0.91        45



In [7]:
matrix = confusion_matrix(y_test, prediction, labels=[0, 1, 2])
fig_cm = go.Figure(go.Heatmap(z=matrix, x=iris.target_names, y=iris.target_names,
                            text=matrix, texttemplate='%{text}', colorscale='Blues',
                            hovertemplate='真实=%{y}<br>预测=%{x}<br>数量=%{z}<extra></extra>'))
fig_cm.update_layout(title='Iris：最后测试集混淆矩阵', xaxis_title='预测类别',
                     yaxis_title='真实类别', height=550, width=1000)
fig_cm.show(renderer='plotly_mimetype')


## 5. 检查错误样本与预处理状态
错误样本可以帮助理解模型边界；如果根据它们继续改进模型，应重新规定最终评估方案。

In [8]:
errors = pd.DataFrame(X_test, columns=iris.feature_names)
errors['原始行ID'] = test_ids
errors['真实类别'] = iris.target_names[y_test]
errors['预测类别'] = iris.target_names[prediction]
assert matrix.sum() == len(test_ids)
assert np.allclose(search.best_estimator_.named_steps['scale'].mean_, X_train.mean(axis=0))
assert np.isfinite(test_macro_f1)
errors.loc[y_test != prediction]


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),原始行ID,真实类别,预测类别
2,6.3,2.8,5.1,1.5,133,virginica,versicolor
30,6.1,2.6,5.6,1.4,134,virginica,versicolor
39,6.7,3.0,5.0,1.7,77,versicolor,virginica
42,4.9,2.5,4.5,1.7,106,virginica,versicolor


## 下一步
先操作教程的软间隔与核实验，再阅读重复外层评估。记录协议、随机种子与参数网格，观察分数波动；一次45样本测试不支持普遍的算法排名。